In [13]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import open3d as o3d
import pandas as pd
from collections import Counter
from tqdm import tqdm

# =========================================================
# 0. CONFIG
# =========================================================
VOXEL_SIZE = 0.005
NUM_POINTS = 1500
MIN_FILE_KB = 200
MIN_FILE_BYTES = MIN_FILE_KB * 1024

ROOT_DIR = r"D:\CVPR_Data\New_PLY"
TARGET_DAY = "Day4_1"
DAY_DIR = os.path.join(ROOT_DIR, TARGET_DAY)

# CSV Outputs
SUMMARY_CSV = rf"C:\Users\spaudel\OneDrive - University of Arkansas System Division of Agriculture\Desktop\Projects_shivapdl\ID_Organized_CVPR\Output_Logs\Day4_1500\sow_level_summary_{TARGET_DAY}_10sows.csv"
VISIT_CSV   = rf"C:\Users\spaudel\OneDrive - University of Arkansas System Division of Agriculture\Desktop\Projects_shivapdl\ID_Organized_CVPR\Output_Logs\Day4_1500\visit_level_logs_{TARGET_DAY}_10sows.csv"
METRICS_TXT = rf"C:\Users\spaudel\OneDrive - University of Arkansas System Division of Agriculture\Desktop\Projects_shivapdl\ID_Organized_CVPR\Output_Logs\Day4_1500\metrics_{TARGET_DAY}_10sows.txt"

# Confusion Matrices & Class Reports
FRAME_CONFUSION_CSV = rf"C:\Users\spaudel\OneDrive - University of Arkansas System Division of Agriculture\Desktop\Projects_shivapdl\ID_Organized_CVPR\Output_Logs\Day4_1500\frame_confusion_{TARGET_DAY}_10sows.csv"
FRAME_CLASS_REPORT_CSV = rf"C:\Users\spaudel\OneDrive - University of Arkansas System Division of Agriculture\Desktop\Projects_shivapdl\ID_Organized_CVPR\Output_Logs\Day4_1500\frame_class_report_{TARGET_DAY}_10sows.csv"
VISIT_CONFUSION_CSV = rf"C:\Users\spaudel\OneDrive - University of Arkansas System Division of Agriculture\Desktop\Projects_shivapdl\ID_Organized_CVPR\Output_Logs\Day4_1500\visit_confusion_{TARGET_DAY}_10sows.csv"
VISIT_CLASS_REPORT_CSV = rf"C:\Users\spaudel\OneDrive - University of Arkansas System Division of Agriculture\Desktop\Projects_shivapdl\ID_Organized_CVPR\Output_Logs\Day4_1500\visit_class_report_{TARGET_DAY}_10sows.csv"

# Logic Thresholds
MIN_VALID_FRAMES_PER_VISIT = 10
CONF_THRESHOLD = 0.99
CONSENSUS_THRESHOLD = 0.50
PRINT_VISIT_LOGS = True

# =========================================================
# 1. MODEL COMPONENTS (PointNet)
# =========================================================
def conv_bn(x, filters):
    x = layers.Conv1D(filters, kernel_size=1, padding="valid")(x)
    x = layers.BatchNormalization(momentum=0.0)(x)
    return layers.Activation("relu")(x)

def dense_bn(x, filters):
    x = layers.Dense(filters)(x)
    x = layers.BatchNormalization(momentum=0.0)(x)
    return layers.Activation("relu")(x)

class OrthogonalRegularizer(keras.regularizers.Regularizer):
    def __init__(self, num_features, l2reg=0.001):
        self.num_features = num_features
        self.l2reg = l2reg
        self.eye = tf.eye(num_features)
    def __call__(self, x):
        x = tf.reshape(x, (-1, self.num_features, self.num_features))
        xxt = tf.tensordot(x, x, axes=(2, 2))
        xxt = tf.reshape(xxt, (-1, self.num_features, self.num_features))
        return tf.reduce_sum(self.l2reg * tf.square(xxt - self.eye))

def tnet(inputs, num_features):
    bias = keras.initializers.Constant(np.eye(num_features).flatten())
    reg = OrthogonalRegularizer(num_features)
    x = conv_bn(inputs, 32); x = conv_bn(x, 64); x = conv_bn(x, 512)
    x = layers.GlobalMaxPooling1D()(x)
    x = dense_bn(x, 256); x = dense_bn(x, 128)
    x = layers.Dense(num_features * num_features, kernel_initializer="zeros",
                     bias_initializer=bias, activity_regularizer=reg)(x)
    mat = layers.Reshape((num_features, num_features))(x)
    return layers.Dot(axes=(2, 1))([inputs, mat])

# =========================================================
# 2. LOAD MODEL
# =========================================================
STABLE_LIST = ['419', '453', '458', '467', '472', '485', '492', '495', '542-533']
CLASS_TO_SOW = {sow: idx for idx, sow in enumerate(STABLE_LIST)}
SOW_FROM_CLASS = {v: k for k, v in CLASS_TO_SOW.items()}
SOW_TO_CLASS = {k: v for k, v in CLASS_TO_SOW.items()}

num_classes = len(STABLE_LIST)
inputs = keras.Input(shape=(NUM_POINTS, 3))
x = tnet(inputs, 3); x = conv_bn(x, 32); x = conv_bn(x, 64); x = tnet(x, 64)
x = conv_bn(x, 128); x = conv_bn(x, 128); x = conv_bn(x, 256)
x = layers.GlobalMaxPooling1D()(x)
x = dense_bn(x, 128); x = layers.Dropout(0.1)(x)
x = dense_bn(x, 128); x = layers.Dropout(0.1)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)
model = keras.Model(inputs, outputs)

WEIGHTS_PATH = r"C:\Users\spaudel\OneDrive - University of Arkansas System Division of Agriculture\Desktop\Projects_shivapdl\ID_Organized_CVPR\Weights\Weight_9_2\9pig_Day32_1500.h5"
model.load_weights(WEIGHTS_PATH)

# =========================================================
# 3. HELPERS
# =========================================================
def process_ply_for_inference(path: str):
    if os.path.getsize(path) < MIN_FILE_BYTES: return None
    try:
        pcd = o3d.io.read_point_cloud(path).voxel_down_sample(voxel_size=VOXEL_SIZE)
        pts = np.asarray(pcd.points)
        if pts.shape[0] < NUM_POINTS: return None
        idx = np.random.choice(pts.shape[0], NUM_POINTS, replace=False)
        pts = pts[idx] - np.mean(pts[idx], axis=0)
        scale = np.max(np.linalg.norm(pts, axis=1))
        return (pts / scale if scale > 0 else pts).astype(np.float32)
    except: return None

def confusion_matrix_np(y_true, y_pred, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        if 0 <= t < num_classes and 0 <= p < num_classes:
            cm[t, p] += 1
    return cm

def save_cm_and_report(cm, cm_path, report_path, labels):
    eps = 1e-12
    tp = np.diag(cm).astype(np.float64)
    fp = (cm.sum(axis=0) - tp).astype(np.float64)
    fn = (cm.sum(axis=1) - tp).astype(np.float64)
    prec = tp / (tp + fp + eps)
    rec = tp / (tp + fn + eps)
    f1 = 2 * prec * rec / (prec + rec + eps)
    sup = cm.sum(axis=1)
    
    pd.DataFrame(cm, index=labels, columns=labels).to_csv(cm_path)
    pd.DataFrame({"SowID": labels, "Precision": prec, "Recall": rec, "F1": f1, "Support": sup}).to_csv(report_path, index=False)
    return np.mean(f1), np.sum(f1 * sup) / np.sum(sup)

# =========================================================
# 4. EVALUATION LOOP
# =========================================================
eval_sows = [s for s in os.listdir(DAY_DIR) if s in STABLE_LIST]
visit_rows = []
summary_rows = []
frame_y_true, frame_y_pred = [], []
visit_y_true, visit_y_pred = [], []

for gt_sow in tqdm(sorted(eval_sows), desc="Processing Sows"):
    gt_class = SOW_TO_CLASS[gt_sow]
    sow_path = os.path.join(DAY_DIR, gt_sow)
    
    sow_valid_visits = 0
    sow_total_visits = 0
    
    for cam in sorted(os.listdir(sow_path)):
        cam_path = os.path.join(sow_path, cam)
        if not os.path.isdir(cam_path): continue
            
        for visit_folder in sorted(os.listdir(cam_path)):
            visit_path = os.path.join(cam_path, visit_folder)
            sow_total_visits += 1
            
            # Predict all frames in visit
            current_visit_preds = []
            files = [f for f in os.listdir(visit_path) if f.lower().endswith(".ply")]
            
            for f in files:
                sample = process_ply_for_inference(os.path.join(visit_path, f))
                if sample is None: continue
                
                prob = model.predict(sample[np.newaxis, ...], verbose=0)
                if np.max(prob) >= CONF_THRESHOLD:
                    pred_cls = int(np.argmax(prob))
                    current_visit_preds.append(pred_cls)
                    # FRAME LEVEL LOGGING
                    frame_y_true.append(gt_class)
                    frame_y_pred.append(pred_cls)

            # VISIT LEVEL LOGGING
            num_valid_f = len(current_visit_preds)
            status = "SKIP"
            pred_sow_id = "N/A"
            correct = False
            
            if num_valid_f >= MIN_VALID_FRAMES_PER_VISIT:
                counts = Counter(current_visit_preds)
                maj_class, maj_count = counts.most_common(1)[0]
                maj_frac = maj_count / num_valid_f
                
                if maj_frac >= CONSENSUS_THRESHOLD:
                    sow_valid_visits += 1
                    pred_sow_id = SOW_FROM_CLASS[maj_class]
                    correct = (str(pred_sow_id) == str(gt_sow))
                    status = "SUCCESS"
                    visit_y_true.append(gt_class)
                    visit_y_pred.append(maj_class)
                else:
                    status = "NO_CONSENSUS"
            
            # Append to detailed visit log
            visit_rows.append({
                "Directory": visit_path,
                "GroundTruth_SowID": gt_sow,
                "Predicted_SowID": pred_sow_id,
                "Status": status,
                "ValidFrames": num_valid_f,
                "IsCorrect": correct
            })

    summary_rows.append({"SowID": gt_sow, "TotalVisits": sow_total_visits, "ValidVisits": sow_valid_visits})

# =========================================================
# 5. SAVE & REPORT
# =========================================================
pd.DataFrame(visit_rows).to_csv(VISIT_CSV, index=False)
pd.DataFrame(summary_rows).to_csv(SUMMARY_CSV, index=False)

# Calculate Metrics
cm_f = confusion_matrix_np(frame_y_true, frame_y_pred, num_classes)
f_macro, f_weighted = save_cm_and_report(cm_f, FRAME_CONFUSION_CSV, FRAME_CLASS_REPORT_CSV, STABLE_LIST)

cm_v = confusion_matrix_np(visit_y_true, visit_y_pred, num_classes)
v_macro, v_weighted = save_cm_and_report(cm_v, VISIT_CONFUSION_CSV, VISIT_CLASS_REPORT_CSV, STABLE_LIST)

print(f"\n--- Efficiency Report ---")
print(f"Frame-Level Accuracy: {np.mean(np.array(frame_y_true) == np.array(frame_y_pred))*100:.2f}%")
print(f"Visit-Level Accuracy: {np.mean(np.array(visit_y_true) == np.array(visit_y_pred))*100:.2f}%")
print(f"Logs saved to: {VISIT_CSV}")

Processing Sows: 100%|██████████| 7/7 [06:03<00:00, 52.00s/it]


--- Efficiency Report ---
Frame-Level Accuracy: 97.73%
Visit-Level Accuracy: 100.00%
Logs saved to: C:\Users\spaudel\OneDrive - University of Arkansas System Division of Agriculture\Desktop\Projects_shivapdl\ID_Organized_CVPR\Output_Logs\Day4_1500\visit_level_logs_Day4_1_10sows.csv
